<a href="https://colab.research.google.com/github/deltorobarba/science/blob/main/google.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build, Test and Evaluate Agents with ADK

###### *⚙️ Setup and Environments*

In [ ]:
# @title Installs
############################################

!pip install --upgrade --quiet "google-cloud-aiplatform[agent_engines,adk,evaluation]"
!pip install --upgrade --quiet google-cloud-secret-manager google-cloud-bigquery anthropic[vertex]
!pip install --quiet a2a-sdk google-adk
!pip install --quiet litellm mcp
!pip show google-adk litellm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 k

In [ ]:
# @title Imports
############################################

import os, asyncio, json, litellm, warnings, vertexai, google.adk, google.auth, google.auth.transport.requests, time
warnings.filterwarnings('ignore')
from google.cloud import storage
from google import genai
from google.genai import types
from google.colab import userdata
from google.genai.types import HttpOptions
from google.genai import types as genai_types
from google.adk.agents import LlmAgent, SequentialAgent, ParallelAgent, Agent
from google.adk.events import Event
from google.adk.tools import google_search, url_context, VertexAiSearchTool, load_memory
from google.adk.sessions import InMemorySessionService, VertexAiSessionService
from google.adk.memory import InMemoryMemoryService, VertexAiMemoryBankService
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from typing import List
import pandas as pd
from pydantic import BaseModel
from vertexai import Client, types, agent_engines
from vertexai.preview import reasoning_engines  # Deployment, Tracing and Telemetry
from google.genai.types import (
    CreateBatchJobConfig,
    CreateCachedContentConfig,
    EmbedContentConfig,
    FunctionDeclaration,
    GenerateContentConfig,
    HarmBlockThreshold,
    HarmCategory,
    Part,
    SafetySetting,
    Tool,)

/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [ ]:
# @title Environmental Variables
############################################

# These tell the underlying google-genai client to use Vertex AI
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = "deltorobarba"
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"
# Required by LiteLLM for Vertex AI non-Gemini models
os.environ["VERTEXAI_PROJECT"] = "deltorobarba"
os.environ["VERTEXAI_LOCATION"] = "us-central1"
# Initialize Vertex AI client
client = vertexai.Client(project="deltorobarba", location="us-central1")
PROJECT_ID="deltorobarba"
LOCATION="us-central1"

In [ ]:
# @title Connect to Google Cloud Project
from google.colab import auth
auth.authenticate_user()

###### ✅ *Agent Pipeline*

In [ ]:
# @title Define Agents and Workflow
############################################

# Vertex AI Search (VAIS)
DATA_STORE_ID = "projects/deltorobarba/locations/global/collections/default_collection/dataStores/deltorobarba-adk_1773523070796"
# Google's out-of-the-box RAG system for information retrieval (for Grounding)
  # Documents are loaded onto Google Cloud Storage bucket (here: 'gs://deltorobarba_adk/')
  # Create new in 'Search app' (Custom search (general) in Vertex AI Search
  # --> Discovery Engine API is underlying engine, includes streamAssist. https://docs.cloud.google.com/generative-ai-app-builder/docs
  # VAIS does (managed): Chunk documents. Choose embedding model. Upload vectors to database. Search that database. Re-rank results.
# https://developers.googleblog.com/building-with-gemini-embedding-2/

# 1. Google Search Agent
web_agent = LlmAgent(
    name='web_researcher',
    model='gemini-2.5-flash',
    description=('Use GoogleSearchTool to find news latest news about Gemini. Return only a list of URLs.'),
    instruction='Search for the latest public news on the topic.',
    tools=[google_search],
    output_key='search_results')

# 2. Vertex AI Search Agent
doc_agent = LlmAgent(
    name='internal_specialist',
    model='gemini-2.5-flash',
    instruction='''Search our internal knowledge base for related documents.
    If the tool returns no results, output EXACTLY: "NO INTERNAL DOCUMENTS FOUND".''',
    tools=[VertexAiSearchTool(data_store_id=DATA_STORE_ID)], # Vertex AI Search with DATA_STORE_ID
    output_key='internal_data')

# 3. Parallel Retrieval Agent
parallel_retrieval = ParallelAgent(
    name="parallel_retrieval",
    sub_agents=[web_agent, doc_agent])

# 4. Web Search Analysis Agent (Processes raw URLs)
search_analyst = LlmAgent(
    name='web_analyst',
    model='gemini-2.5-flash',
    instruction='''Use the url_context tool to read the URLs in {search_results}.
    Extract only technical specifications and performance benchmarks.''',
    tools=[url_context],
    output_key='web_analysis')

# 5. Deep Analysis Agent ("Brain")
analyst = LlmAgent(
    name='analyst',
    #model=deepseek,                   # <----- Be careful with region of 3P models for deployed version
    model='gemini-2.5-pro',
    description=('Compares Gemini 3 vs GPT-4o.'),
    instruction='''
    Check your memory for any user-specific instructions or past preferences.
    Compare public findings ({web_analysis}) against our private data ({internal_data}).
    Identify unique Gemini 3 features that GPT-4o lacks.
    ''',
    tools=[load_memory],  # Recall information from Vertex AI Memory Services!
    output_key='final_comparison')

# 6. CEO Summarizer Agent
summarizer = LlmAgent(
    name='summarizer',
    model='gemini-2.5-flash',
    description=('Draft concise summary'),
    instruction='''
    You are an Executive Intelligence Analyst.
    Summarize the findings in {final_comparison} into 5 punchy bullets for the CEO.
    Ensure you highlight contradictions between internal and public data.
    Max 2000 characters.
    ''',
    output_key='summary' # not used in deployed version)

# 7. Root Orchestrator (Master Flow)
root_agent = SequentialAgent(
    name='researcher',
    # no model needed
    # no instructions needed, it just moves data from 0 -> 1 -> 2 -> 3
    sub_agents=[parallel_retrieval, search_analyst, analyst, summarizer],)

In [ ]:
# @title Deploy Pipeline on Agent Engine (⚠️ --> once only!)
############################################

"""
Permissions: Take the AI Platform Reasoning Engine Service Agent that the agent uses:
'service-892203813305@gcp-sa-aiplatform-re.iam.gserviceaccount.com'.
Go to IAM, enable 'Include Google-provided role grants' and add role: 'Discovery Engine Viewer', 'Vertex AI User' and 'Cloud Trace Agent' (write)
Troubleshooting: https://docs.cloud.google.com/agent-builder/agent-engine/troubleshooting/use

Traceing and Logging:
* Cloud Logging API in GCP project aktivieren
* Telemetry API in GCP project aktivieren
* Add role 'Cloud Trace Agent' (write) to service account
* https://docs.cloud.google.com/agent-builder/agent-engine/manage/tracing
* https://docs.cloud.google.com/agent-builder/agent-engine/deploy
"""

# Create GCS bucket first if you haven't already
STAGING_BUCKET = "gs://deltorobarba-agent-staging"

# https://google.github.io/adk-docs/deploy/agent-engine/deploy/#setup-cloud-project
# https://docs.cloud.google.com/agent-builder/agent-engine/develop/custom
# Environmental Variables for Traceing (https://docs.cloud.google.com/agent-builder/agent-engine/deploy)
# Set env variables (https://docs.cloud.google.com/agent-builder/agent-engine/manage/tracing)
# OpenTelemetry (OTel) as native standard for observability across Google Cloud: Cloud Trace, Logging, and Monitoring
  # ADK and Agent Engine are OpenTelemetry-native
  # https://docs.cloud.google.com/trace/docs/finding-traces
  # GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY) allows system to use OpenTelemetry libraries to package data and send it via OTLP endpoint
custom_env_vars = {
    "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "true",
    "OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT": "true",
}

# 1. Initialize the client
client = vertexai.Client(project="deltorobarba", location="us-central1")

# 2. Wrap the agent
adk_app = reasoning_engines.AdkApp(
    agent=root_agent,
    enable_tracing=True,
    env_vars=custom_env_vars
)

# 3. Deploy
# 'client.agent_engines.update' for updating existing one
remote_agent = client.agent_engines.create(
    agent=adk_app,
    config={
        # This tells Vertex AI where to upload your agent's code
        "display_name":"Agent Engine for Agent Pipeline",
        "staging_bucket": STAGING_BUCKET,
        "requirements": [
            "google-cloud-aiplatform[agent_engines,adk]>=1.135",
            "google-adk>=1.23.0",
            "google-cloud-trace"
        ],
        "env_vars":custom_env_vars,
        "context_spec": {
            "memory_bank_config": {
                "generation_config": {
                    "model": f"projects/deltorobarba/locations/us-central1/publishers/google/models/gemini-2.5-flash"
                }
            }
        }
    },
)

INFO:vertexai_genai.agentengines:Identified the following requirements: {'google-cloud-aiplatform': '1.141.0', 'pydantic': '2.12.3', 'cloudpickle': '3.1.2'}
INFO:vertexai_genai.agentengines:The following requirements are appended: {'pydantic==2.12.3', 'cloudpickle==3.1.2'}
INFO:vertexai_genai.agentengines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]>=1.135', 'google-adk>=1.23.0', 'google-cloud-trace', 'pydantic==2.12.3', 'cloudpickle==3.1.2']
INFO:vertexai_genai.agentengines:Using bucket deltorobarba-agent-staging
INFO:vertexai_genai.agentengines:Wrote to gs://deltorobarba-agent-staging/agent_engine/agent_engine.pkl
INFO:vertexai_genai.agentengines:Writing to gs://deltorobarba-agent-staging/agent_engine/requirements.txt
INFO:vertexai_genai.agentengines:Creating in-memory tarfile of extra_packages
INFO:vertexai_genai.agentengines:Writing to gs://deltorobarba-agent-staging/agent_engine/dependencies.tar.gz
INFO:vertexai_genai.agentengines:Using agent framew

In [ ]:
# @title Test Agent ====> Single Turn Query
############################################

# 1. Initialize the client
client = vertexai.Client(project="deltorobarba", location="us-central1")

# 2. Remote app handle
RESOURCE_NAME = "projects/622694106880/locations/us-central1/reasoningEngines/5366354515849117696"
remote_app = client.agent_engines.get(name=RESOURCE_NAME)

async def run_remote_test():
    print(f"Connecting to: {RESOURCE_NAME}...")
    user_id = "deltorobarba"

    # 3. Create session and grab the 'id' key specifically 🔑
    raw_session = await remote_app.async_create_session(user_id=user_id)

    # Path A: First single-turn query:
    session_id = raw_session.get('id')
    if not session_id:
        print(f"❌ Error: ID not found in response: {raw_session}")
        return

    # Path B: Follow-up single-turn query (explicitly reuse Session ID from  first run!)
    #print(f"Resuming Session on: {RESOURCE_NAME}...")
    #session_id = "1107804831667453952"        #  <-------- copy/paste existing session ID !!!

    print(f"✅ Managed Session ID: {session_id}")

    # 4. Stream the query (nur Summarizer-Events anzeigen) 🌊
    async for event in remote_app.async_stream_query(
        user_id=user_id,
        session_id=session_id,
        #message="Uluru bank uses ChatGPT. The annual revenue of Uluru bank is 500 Mio USD. Search news on Gemini 3 and compare it to GPT-4o."
        message="I lead 100 employees. What are the latest 2026 news on Gemini 3?"
    ):
        # Filtern nach dem Author 'summarizer' 🔍
        # if 'content' in event: # print every agent's output
        if event.get('author') == 'summarizer' and 'content' in event:
            for part in event['content'].get('parts', []):
                if 'text' in part:
                    print(f"\n--- Agent Output (Summary Agent) ---\n{part['text']}")

    # 5. Trigger Memory Extraction 🧠
    print(f"\n[System] Fetching session state for extraction...")
    current_session = await remote_app.async_get_session(user_id=user_id, session_id=session_id)

    # Check for events to ensure history is captured
    # Docs: https://google.github.io/adk-docs/events/#extracting-key-information
    if current_session.get('events'):
        print(f"✅ Found {len(current_session['events'])} events. Sending to Memory Bank...")
        # Pass the dictionary directly to the memory service
        await remote_app.async_add_session_to_memory(session=current_session)
        print("\n[Done] Memory extraction triggered! Check 'Gemerkte Informationen' in 1-2 mins.")
    else:
        print("\n❌ Error: Session history was empty. No memories extracted.")

# Execute
await run_remote_test()

Connecting to: projects/622694106880/locations/us-central1/reasoningEngines/5366354515849117696...
✅ Managed Session ID: 5806649411706552320

--- Agent Output (Summary Agent) ---
Here are the latest 2026 updates on Gemini 3, synthesized for you:

*   **Core Intelligence Upgrades:** Gemini 3.1 Pro, Flash, and Deep Think rolled out in early 2026, delivering enhanced reasoning, "PhD-level speed," and specialized capabilities for complex scientific problems.
*   **Broad Adoption & Integration:** General Motors is integrating Gemini into millions of vehicles, while Google aims for "Proactive Assistance" features anticipating user needs, cementing its presence across daily life.
*   **Ecosystem Maturity:** The platform is now a mature, production-ready stack, deeply integrated across Google's product ecosystem, from Search to Vertex AI, ensuring widespread access and utility.
*   **Gemini 4 Vision:** An early look at Gemini 4 is expected at Google I/O in May 2026, promising an "AI employee" 

In [ ]:
# @title Test Agent ====> Multi-Turn Query
############################################

async def run_multi_turn_test():
    print(f"Connecting to: {RESOURCE_NAME}...")
    user_id = "deltorobarba"

    # --- Turn 1: Initial Query (Creates the Session) ---
    raw_session = await remote_app.async_create_session(user_id=user_id)
    session_id = raw_session.get('id')
    print(f"✅ Session Created: {session_id}")

    # FIRST MESSAGE
    await execute_query(user_id, session_id, "I am an astronaut. Uluru bank uses ChatGPT. Annual revenue is 500 Mio USD. Compare Gemini 3 to GPT-4o.")

    # --- Turn 2: Follow-up Query (Reuses the Session ID) ---
    print(f"\n--- Starting Follow-up Query in Session: {session_id} ---")

    # SECOND MESSAGE (re-uses same session_id variable)
    await execute_query(user_id, session_id, "Based on that revenue, which model is more cost-effective for us?")

    # --- Step 5: Final Memory Extraction ---
    # Fetching the session now will show events from BOTH queries.
    current_session = await remote_app.async_get_session(user_id=user_id, session_id=session_id)
    if current_session.get('events'):
        print(f"✅ Found {len(current_session['events'])} total events. Extracting to Memory Bank...")
        await remote_app.async_add_session_to_memory(session=current_session)

# Helper function to keep the code clean
async def execute_query(user_id, session_id, message):
    async for event in remote_app.async_stream_query(
        user_id=user_id,
        session_id=session_id,
        message=message
    ):
        if event.get('author') == 'summarizer' and 'content' in event:
            for part in event['content'].get('parts', []):
                if 'text' in part:
                    print(f"\n[Summarizer]: {part['text']}")

await run_multi_turn_test()

Connecting to: projects/622694106880/locations/us-central1/reasoningEngines/5366354515849117696...
✅ Session Created: 8177665422900854784

--- Starting Follow-up Query in Session: 8177665422900854784 ---

[Summarizer]: Here are 5 punchy bullets for the CEO:

*   **Internal Intel:** A potential "Gemini 3" is internally described with "quantum telepathy," capable of processing "quantum-native data" and "entangled information."
*   **GPT-4o Lacks This:** These "quantum" capabilities are highly advanced and utterly absent from current models like GPT-4o, which focuses on multimodal (text, audio, vision) excellence.
*   **Key Contradiction:** This "quantum telepathy" feature is *only* from internal documents; there is *no public information* or official release confirming such a "Gemini 3."
*   **Public Reality:** Google's *actual* publicly discussed Gemini 3.x models (3.1 Pro/Flash Lite) are advanced multimodal AIs, but *do not* feature any quantum telepathy.
*   **Action:** The "quantum" 

###### ✅ *Evaluation*

In [ ]:
# @title Agent Evaluation
# Trace-based: no agent_info, no fictional tool declarations
# ============================================================

# Based on: https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-agents

# Traditional model eval only looks at the final string output. Like Auto SxS with Autorater for relative evaluation.
# Trend moves to model-as-a-judge (serverless behind client.evals.create_evaluation_run)
  # with adaptive rubrics (types.RubricMetric), like Unit Test for agents)
# We are calling client.evals.run_inference which returns traces - important piece of agent evaluation.
# Agent Evaluation evaluates the entire execution lifecycle captured in the trace:
  # Reasoning: Did the agent's internal thought process make sense?
  # Tool Call Accuracy: Did it call the right tool with the correct parameters?
  # Grounding: Is the final answer actually supported by the data returned from those tool calls?

import vertexai
from google.cloud import storage
from google.genai import types as genai_types
from vertexai import Client
import time
import pandas as pd
from vertexai import types

# ---- 1. Config ----------------------------------------------
PROJECT_ID = "deltorobarba"
LOCATION   = "us-central1"
AGENT      = "projects/622694106880/locations/us-central1/reasoningEngines/5366354515849117696"
GCS_DEST   = "gs://deltorobarba-agent-staging"

# ---- 2. Init SDK client -------------------------------------
vertexai.init(project=PROJECT_ID, location=LOCATION)
client = Client(
    project=PROJECT_ID,
    location=LOCATION,
    http_options=genai_types.HttpOptions(api_version="v1beta1"),
)

# ---- 3. Build eval dataset ------------------------------
session_inputs = types.evals.SessionInput(user_id="eval_user", state={})

agent_prompts = [
    "What are the latest 2026 news on Gemini 3?",
    "Find me new feature updates for Gemini 3!",
]

agent_dataset = pd.DataFrame({
    "prompt": agent_prompts,
    "session_inputs": [session_inputs] * len(agent_prompts),
})

# ---- 4. Run inference -----------
traces = client.evals.run_inference(
    agent=AGENT,
    src=agent_dataset,
)
traces.show()

# ---- 5. Create Evaluation Report --------------------------------------------

eval_result = client.evals.create_evaluation_run(
    dataset=traces,                # run_inference output works as the dataset
    agent=AGENT,
    metrics=[
        types.RubricMetric.FINAL_RESPONSE_QUALITY,
        types.RubricMetric.HALLUCINATION,
        types.RubricMetric.SAFETY,
        # TOOL_USE_QUALITY omitted — without agent_info it has nothing to score against
    ],
    dest=GCS_DEST,
)

import time
while eval_result.state not in {"SUCCEEDED", "FAILED", "CANCELLED"}:
    time.sleep(5)
    eval_result = client.evals.get_evaluation_run(name=eval_result.name)

eval_result = client.evals.get_evaluation_run(
    name=eval_result.name, include_evaluation_items=True
    # name=EVAL_RUN_ID, include_evaluation_items=True # if we want to retrieve previous results
)
eval_result.show()

In [ ]:
# @title Model Evaluation

# https://docs.cloud.google.com/vertex-ai/generative-ai/docs/models/evaluation-overview
# https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/evaluating_third_party_llms_vertex_ai_gen_ai_eval_sdk.ipynb
# https://cloud.google.com/blog/products/ai-machine-learning/evaluate-ai-models-with-vertex-ai--llm-comparator
# https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/quick_start_gen_ai_eval.ipynb

"""
#####################################
# Model Evaluation (Single Model)
#####################################
MAAS_MODEL_ID = "vertex_ai/deepseek-ai/deepseek-v3.2-maas"
# fmt: on

# Select a MaaS model. Remember to check regional availability!
# model = "deepseek-ai/deepseek-r1-0528-maas" # Example model
# model = "meta/llama-3.1-70b-instruct-maas"  # Example model
# model = "meta/llama-4-maverick-17b-128e-instruct-maas"   # Example model in us-east5
# model = "claude-3-5-haiku"  # Example model in us-east5
# model = "qwen/qwen3-coder-480b-a35b-instruct-maas"  # Example model in us-south1

# Run inference on MaaS model to create eval dataset
print(f"--- Running Inference for MaaS Model: {MAAS_MODEL_ID} ---")
eval_dataset = client.evals.run_inference(
    model=MAAS_MODEL_ID,
    src="gs://vertex-evaluation-llm-dataset-us-central1/genai_eval_sdk/test_prompts.jsonl",
)
eval_dataset.show()

# Evaluate result
print(f"\n--- Running Evaluation for MaaS Model: {MAAS_MODEL_ID} ---")
maas_eval_result = client.evals.evaluate(
    dataset=eval_dataset,
    metrics=[
        types.RubricMetric.GENERAL_QUALITY,
        types.RubricMetric.INSTRUCTION_FOLLOWING,
        types.Metric(name="rouge_1"),
        types.Metric(name="bleu"),
    ],
)
maas_eval_result.show()
"""

#####################################
# Model Evaluation (Multi Model)
#####################################
# ----- Model Comparison Evaluation: Get data -----
import pandas as pd

prompts_df = pd.DataFrame(
    {
        "prompt": [
            "Explain the difference between correlation and causation, and provide a real-world example where confusing the two could lead to poor decision-making.",
            "Write a Python function that finds the longest palindromic substring in a given string. Include comments explaining your approach and time complexity.",
            "A train leaves Station A at 9:00 AM traveling at 60 mph toward Station B. Another train leaves Station B at 10:00 AM traveling at 80 mph toward Station A. If the stations are 280 miles apart, at what time do the trains meet?",
            "Analyze the ethical implications of using AI in hiring decisions. Present arguments from multiple perspectives and discuss potential safeguards.",
            "Translate the following sentence to French, Spanish, and German, then explain any cultural nuances that might affect the translation: 'The early bird catches the worm, but the second mouse gets the cheese.'",
            "Create a short story (200 words) that includes these elements: a mysterious package, a lighthouse keeper, and a revelation that changes everything. The story should have a clear beginning, middle, and end.",
            "Compare and contrast the economic theories of Adam Smith and Karl Marx. How would each theorist likely view modern gig economy platforms like Uber?",
            "Debug this code and explain what's wrong: def fibonacci(n): if n <= 1: return n else: return fibonacci(n-1) + fibonacci(n-2) + fibonacci(n-3)",
            "You're a manager and an employee consistently delivers excellent work but is frequently late to meetings. Write a constructive feedback message addressing this issue while maintaining morale.",
            "Explain how transformer architecture works in machine learning to someone with basic programming knowledge but no ML background. Use an analogy to clarify the concept of attention mechanisms.",
        ]
    }
)

data_with_rubrics = client.evals.generate_rubrics(
    src=prompts_df,
    rubric_group_name="general_quality_rubrics",
    predefined_spec_name=types.RubricMetric.GENERAL_QUALITY,
)

# ----- Evaluate result -----
print(f"\n--- Running Evaluation for MaaS Models")

import os
# fmt: off
PROJECT_ID = "deltorobarba"
if not PROJECT_ID or PROJECT_ID == "deltorobarba":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))
LOCATION= "global"
# fmt: on
LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", LOCATION)

location = "global"
project = "deltorobarba"


# --- Model 1: Qwen---
qwen_dataset = client.evals.run_inference(
    model="vertex_ai/qwen/qwen3-next-80b-a3b-thinking-maas",
    src=data_with_rubrics,
)

# --- Model 2: DeepSeek MAAS Model ---
deepseek_dataset = client.evals.run_inference(
    model="vertex_ai/deepseek-ai/deepseek-v3.2-maas",
    src=data_with_rubrics,
)

# --- Run Comparison Evaluation ---
comparison_eval_result = client.evals.evaluate(
    dataset=[qwen_dataset, deepseek_dataset],
    metrics=[
        types.RubricMetric.GENERAL_QUALITY(rubric_group_name="general_quality_rubrics")
    ],
)
comparison_eval_result.show()

LiteLLM Inference (vertex_ai/qwen/qwen3-next-80b-a3b-thinking-maas):   0%|          | 0/10 [00:00<?, ?it/s]15:38:16 - LiteLLM:WARNING: vertex_llm_base.py:67 - Vertex AI model 'qwen/qwen3-next-80b-a3b-thinking-maas' does not support region 'us-central1' (supported: ['global']). Routing to 'global'.
15:38:16 - LiteLLM:WARNING: vertex_llm_base.py:67 - Vertex AI model 'qwen/qwen3-next-80b-a3b-thinking-maas' does not support region 'us-central1' (supported: ['global']). Routing to 'global'.
15:38:16 - LiteLLM:WARNING: vertex_llm_base.py:67 - Vertex AI model 'qwen/qwen3-next-80b-a3b-thinking-maas' does not support region 'us-central1' (supported: ['global']). Routing to 'global'.
15:38:16 - LiteLLM:WARNING: vertex_llm_base.py:67 - Vertex AI model 'qwen/qwen3-next-80b-a3b-thinking-maas' does not support region 'us-central1' (supported: ['global']). Routing to 'global'.
15:38:16 - LiteLLM:WARNING: vertex_llm_base.py:67 - Vertex AI model 'qwen/qwen3-next-80b-a3b-thinking-maas' does not support 

###### ✅ *Model Garden*

In [ ]:
# @title Google SDK

# https://github.com/se02035/local-llm-proxy it helps to spin up a litellm proxy with local ollama. also supports a public endpoint using ngrok

# Locations: https://docs.cloud.google.com/vertex-ai/generative-ai/docs/learn/locations
# Google Models: https://docs.cloud.google.com/vertex-ai/generative-ai/docs/models
# Versions: https://docs.cloud.google.com/vertex-ai/generative-ai/docs/learn/model-versions

# client = genai.Client(http_options=HttpOptions(api_version="v1")) --> only for Google models directly
# https://github.com/GoogleCloudPlatform/generative-ai/blob/main/sdk/intro_genai_sdk.ipynb
# https://docs.cloud.google.com/vertex-ai/generative-ai/docs/open-models/use-maas
from google import genai
from google.genai.types import GenerateContentConfig
from google.genai import types

PROJECT_ID="deltorobarba"
LOCATION = "global"
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

gemini    = "google/gemini-3.1-pro-preview"
gemma     = "google/gemma-4-26b-a4b-it-maas"
grok      = "xai/grok-4.1-fast-reasoning"
deepseek  = "deepseek-ai/deepseek-v3.2-maas"
kimi      = "moonshotai/kimi-k2-thinking-maas"
qwen      = "qwen/qwen3-next-80b-a3b-thinking-maas"
minimax   = "minimaxai/minimax-m2-maas"
glm       = "zai-org/glm-5-maas"
llama     = "meta/llama-3.3-70b-instruct-maas"   # LOCATION = "us-central1"
claude    = "anthropic/claude-opus-4-7"

system_instruction = """You are a helpful machine learning advisor and answer briefly."""
prompt = """Hi, which LLM are you?"""

generate_content_config = types.GenerateContentConfig(
    temperature=0.4,
    top_p=0.95,
    top_k=20,
    candidate_count=1,
    seed=5,
    max_output_tokens=60,
    stop_sequences=["STOP!"],
    presence_penalty=0.0,
    frequency_penalty=0.0,
    system_instruction=system_instruction,)

response = client.models.generate_content(
    model=gemma,                       # <--- ⚠️ Define model here
    contents=prompt,
    config=generate_content_config,
)
print("✅ Success")
print(response.text)

✅ Success
I am a large language model, trained by Google.


In [ ]:
# @title OpenAI SDK
#################################################

from openai import OpenAI

PROJECT  = "deltorobarba"
LOCATION = "global"

# ⚠️ Accept User Agreements per Model!
grok = "xai/grok-4.1-fast-reasoning"        # ✅ https://docs.cloud.google.com/vertex-ai/generative-ai/docs/partner-models/grok/grok-4-1-fast
gemma = "google/gemma-4-26b-a4b-it-maas"    # ✅ https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/google/gemma-4-26b-a4b-it
gemini = "google/gemini-3.1-pro-preview"    # ✅ https://docs.cloud.google.com/vertex-ai/generative-ai/docs/models/gemini/3-1-pro
deepseek = "deepseek-ai/deepseek-v3.2-maas" # ✅ https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/deepseek/deepseek-v32

# Get access token from Application Default Credentials (run `gcloud auth application-default login` once beforehand)
credentials, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
credentials.refresh(google.auth.transport.requests.Request())

# Point OpenAI SDK at Vertex's OpenAI-compatible endpoint
client = OpenAI(
    base_url=f"https://aiplatform.googleapis.com/v1/projects/{PROJECT}/locations/{LOCATION}/endpoints/openapi",
    api_key=credentials.token,
)

# Call selectec model like any OpenAI model
response = client.chat.completions.create(
    model=deepseek,
    messages=[{"role": "user", "content": "Hi, which LLM are you?"}],
    temperature=0.2,
)

print("✅ Success")
print(response.choices[0].message.content)

✅ Success
你好！我是DeepSeek，由深度求索公司创造的AI助手！😊

我是一个纯文本模型，具有以下特点：
- 完全免费使用
- 支持128K上下文长度
- 可以处理文件上传（图像、txt、pdf、ppt、word、excel等）
- 支持联网搜索（需要手动开启）
- 可通过官方应用商店下载App使用

虽然我不支持多模态识别，但可以读取上传文件中的文字信息来帮助你。我的知识截止到2024年7月，会尽我所能为你提供准确、有用的回答！

有什么问题我可以帮你解答吗？我很乐意为你提供帮助！✨


In [ ]:
# @title LiteLLM
#################################################

# DeepSeek V3.2 - https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/deepseek/deepseek-v32
deepseek = LiteLlm(
    model="vertex_ai/deepseek-ai/deepseek-v3.2-maas",
    vertex_project="deltorobarba",
    vertex_location="global")

# Kimi K2 - https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/kimi/kimi-k2-thinking
kimi = LiteLlm(
    model="vertex_ai/kimi/kimi-k2-thinking-maas",
    vertex_project="deltorobarba",
    vertex_location="global")

# Qwen 3 - https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/qwen/qwen3-next-thinking
qwen = LiteLlm(
    model="vertex_ai/qwen/qwen3-next-80b-a3b-thinking-maas",
    vertex_project="deltorobarba",
    vertex_location="global")

# MiniMax - https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/minimax
minimax = LiteLlm(
    model="vertex_ai/minimaxai/minimax-m2-maas",
    vertex_project="deltorobarba",
    vertex_location="global")

# GLM 5 - https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/zaiorg/glm-5
glm = LiteLlm(
    model="vertex_ai/zai-org/glm-5-maas",
    vertex_project="deltorobarba",
    vertex_location="global")

# ⚠️ ----- Quick Test on Model Availability -----
deepseek = "vertex_ai/deepseek-ai/deepseek-v3.2-maas"     # ✅
kimi = "vertex_ai/moonshotai/kimi-k2-thinking-maas"       # ✅
qwen = "vertex_ai/qwen/qwen3-next-80b-a3b-thinking-maas"  # ✅
minimax = "vertex_ai/minimaxai/minimax-m2-maas"           # ✅
glm ="vertex_ai/zai-org/glm-5-maas"                       # ✅

project = "deltorobarba"
location = "global"
question = "Hi"

try:
    response = litellm.completion(
        model=deepseek,                                    # <--- ⚠️ Add model here
        messages=[{"role": "user", "content": question}],
        temperature=0.2,
        #max_tokens=264,
        vertex_project=project,
        vertex_location=location)
    print("✅ Success: The model is reachable!")
    print(response.choices[0].message.content)
except Exception as e:
    print(f"❌ Failed: Connectivity or Model Name error: {e}")

✅ Success: The model is reachable!
你好！👋 很高兴见到你！

有什么我可以帮助你的吗？无论是回答问题、聊天、协助解决问题，还是其他任何事情，我都很乐意为你提供帮助！😊


In [ ]:
# @title Vertex AI - Deployment Recommendation
from vertexai import model_garden

# google/gemma-4-26B-A4B-it
# google/gemma-4-31B-it

# Initialize for your project/location
# model_id can be "google/gemma-4-26B-A4B-it" or "google/gemma-4-31B-it"
model = model_garden.OpenModel("google/gemma-4-26B-A4B-it")

# Get the recommended/verified deployment options
deploy_options = model.list_deploy_options()
print(deploy_options)

[model_display_name: "google/gemma-4-26B-A4B-it"
container_spec {
  image_uri: "us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/pytorch-vllm-serve:20251205_0916_RC01"
  args: "python"
  args: "-m"
  args: "vllm.entrypoints.api_server"
  args: "--host=0.0.0.0"
  args: "--port=8080"
  args: "--swap-space=16"
  args: "--model=google/gemma-4-26B-A4B-it"
  args: "--revision=7d4c97e54145f8ffd1a4dd1b4986a5015a517842"
  args: "--max-model-len=2048"
  args: "--gpu-memory-utilization=0.9"
  args: "--enforce-eager"
  args: "--tensor-parallel-size=1"
  args: "--enable-chunked-prefill"
  env {
    name: "DEPLOY_SOURCE"
    value: "UI_HF_VERIFIED_MODEL"
  }
  env {
    name: "MODEL_ID"
    value: "google/gemma-4-26B-A4B-it"
  }
  ports {
    container_port: 8080
  }
  predict_route: "/generate"
  health_route: "/ping"
}
dedicated_resources {
  machine_spec {
    machine_type: "a3-highgpu-1g"
    accelerator_type: NVIDIA_H100_80GB
    accelerator_count: 1
  }
  max_replica_count: 1
}
d

In [ ]:
# @title Vertex AI - Deployment Script

"""
Recommendation: NVIDIA RTX Pro 6000 (Blackwell) with VRAM	96 GB
We prefer NVIDIA L4 (Ada Lovelace) with VRAM 24 GB. Capacity: The L4 has 24 GB of GDDR6 memory.
The Problem: The Gemma 4 26B model in full precision (BF16) requires roughly 52 GB of VRAM just to load the weights. It will not fit on a single L4 in that state.
The Solution (Quantization): To run this on an L4, you must use 4-bit quantization (like AWQ or INT4). In 4-bit mode, the weights take up only ~15–18 GB, which fits comfortably on a single L4.
The Catch: You will likely need to reduce your context window significantly (e.g., from 256K down to 8K or 16K) to stay within the L4's 24GB limit.
"""

!pip install --upgrade google-cloud-aiplatform
#gcloud auth application-default login

"""
import vertexai
from vertexai import model_garden

vertexai.init(project="lunar-352813", location="europe-west4")

model = model_garden.OpenModel("google/gemma4@gemma-4-26b-a4b-it")
endpoint = model.deploy(
  accept_eula=True,
  machine_type="g4-standard-48",
  accelerator_type="NVIDIA_RTX_PRO_6000",
  accelerator_count=1,
  serving_container_image_uri="us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/pytorch-vllm-serve:gemma4",
  endpoint_display_name="gemma-4-26b-a4b-it-mg-one-click-deploy",
  model_display_name="gemma-4-26b-a4b-it-1777568163447",
  use_dedicated_endpoint=True,
  reservation_affinity_type="NO_RESERVATION",
)
"""

import vertexai
from vertexai import model_garden

vertexai.init(project="lunar-352813", location="europe-west4")

model = model_garden.OpenModel("google/gemma4@gemma-4-31b-it")
endpoint = model.deploy(
  accept_eula=True,
  machine_type="g4-standard-48",
  accelerator_type="NVIDIA_RTX_PRO_6000",
  accelerator_count=1,
  serving_container_image_uri="us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/pytorch-vllm-serve:gemma4",
  endpoint_display_name="gemma-4-31b-it-mg-one-click-deploy",
  model_display_name="gemma-4-31b-it-1777570278133",
  use_dedicated_endpoint=True,
  reservation_affinity_type="NO_RESERVATION",
)